# Part3_alu_coverage_guided

### Setup Paths and Imports
Import required libraries, define the RTL/specification paths, and create the output directory.

In [12]:
from pathlib import Path
import json
import subprocess
import re
import os
import sys
from typing import Dict, Any, List, Optional

ROOT = Path(".")
RTL_PATH = ROOT / "coverage_guided_alu.v"
SPEC_PATH = ROOT / "coverage_guided_alu_spec.txt"

OUTPUT_DIR = ROOT / "alu_coverage_output"
OUTPUT_DIR.mkdir(exist_ok=True)

print("RTL exists:", RTL_PATH.exists())
print("Spec exists:", SPEC_PATH.exists())
print("Output dir:", OUTPUT_DIR)

RTL exists: True
Spec exists: True
Output dir: alu_coverage_output


### Load ALU Specification and RTL
Read and display the natural-language ALU specification and Verilog source.

In [20]:
spec_text = SPEC_PATH.read_text()
rtl_text = RTL_PATH.read_text()

print("=== SPEC ===")
print(spec_text)

print("\n=== RTL ===")
print(rtl_text)

=== SPEC ===
Coverage-Guided 8-bit ALU - Natural-Language Specification

Inputs:
- a: 8-bit operand
- b: 8-bit operand
- op: 4-bit operation selector

Outputs:
- y: 8-bit result
- zero: 1 if y is 0, else 0
- negative: y[7]
- carry: carry-out for ADD; for SUB, 1 means no unsigned borrow; 0 otherwise
- overflow: signed two's-complement overflow for ADD and SUB; 0 otherwise

Operations:
- op=0x0 ADD: y = a + b
- op=0x1 SUB: y = a - b
- op=0x2 AND: y = a & b
- op=0x3 OR:  y = a | b
- op=0x4 XOR: y = a ^ b
- op=0x5 NOT: y = ~a (b is ignored)
- op=0x6 SLL: y = a << b[2:0]
- op=0x7 SRL: y = a >> b[2:0]
- op=0x8 SRA: y = arithmetic right shift of signed a by b[2:0]
- op=0x9 SLT: signed comparison; y=1 if signed(a) < signed(b), else y=0
- op=0xA..0xF: y=0, carry=0, overflow=0

Corner cases worth exercising include zero results, negative results, ADD carry-out,
ADD/SUB signed overflow, SUB borrow/no-borrow, shift amounts 0/1/7, arithmetic right
shift of negative values, signed SLT around 0x80/0x

### Configure Portkey AI Gateway
Set the NYU Portkey API configuration and select the LLM model.

In [ ]:
# Portkey AI gateway settings
os.environ['PORTKEY_BASE_URL'] = 'https://ai-gateway.apps.cloud.rt.nyu.edu/v1'
os.environ['PORTKEY_API_KEY'] = ''

# Model to use. Other options:
#   '@vertexai/anthropic.claude-sonnet-4-6', '@vertexai/anthropic.claude-opus-4-6',
#   '@vertexai/gemini-2.5-pro', '@vertexai/gemini-2.5-flash',
#   '@gpt-4o/gpt-4o', '@gpt-4o-mini/gpt-4o-mini', '@o4-mini/o4-mini'
os.environ['PORTKEY_MODEL'] = '@vertexai/anthropic.claude-sonnet-5-5'

# Check if API key is set
if os.environ['PORTKEY_API_KEY']:
    print("✓ Portkey API key is configured")
    print(f"  Model: {os.environ['PORTKEY_MODEL']}")
else:
    print("⚠ Warning: Portkey API key not set. Running in demo mode.")
    print("  Set your API key with: os.environ['PORTKEY_API_KEY'] = 'your-key'")


✓ Portkey API key is configured
  Model: @vertexai/anthropic.claude-sonnet-5-5


### Create Output Directory
Ensure the ALU coverage output directory exists before generating artifacts.

In [22]:
output_dir = "alu_coverage_output"
os.makedirs(output_dir, exist_ok=True)
print(f"✓ Output directory created: {output_dir}")

✓ Output directory created: alu_coverage_output


### Define the LLM Client
Create a reusable Python client for sending prompts through the NYU Portkey gateway.

In [23]:
"""
LLM Client for interacting with language models.
Uses the NYU Portkey AI gateway, which gives access to multiple providers (Anthropic, Google, OpenAI, etc.)
"""

import os
import json
from typing import Optional, Dict, Any
from portkey_ai import Portkey


class LLMClient:
    """Client for interacting with LLM APIs."""

    def __init__(self, api_key: Optional[str] = None, model: str = "@vertexai/anthropic.claude-sonnet-5-5", provider: str = "portkey"):
        """
        Initialize LLM client.

        Args:
            api_key: API key for the LLM provider (if None, reads from environment)
            model: Model name to use
            provider: LLM provider ('portkey')
        """
        self.provider = provider
        self.model = model
        self.api_key = api_key or os.getenv("PORTKEY_API_KEY")
        self.client = Portkey(
            base_url=os.getenv("PORTKEY_BASE_URL", "https://ai-gateway.apps.cloud.rt.nyu.edu/v1"),
            api_key=self.api_key,
        )
        # Some models (e.g. Claude Sonnet 5.5) reject the temperature parameter
        self.use_temperature = True

    def generate(self, prompt: str, system_prompt: Optional[str] = None,
                 temperature: float = 0.7, max_tokens: int = 4000) -> str:
        """
        Generate text using the LLM.

        Args:
            prompt: User prompt
            system_prompt: System prompt for the model
            temperature: Sampling temperature (0-1)
            max_tokens: Maximum tokens to generate

        Returns:
            Generated text response
        """
        try:
            if self.provider == "portkey":
                messages = []
                if system_prompt:
                    messages.append({"role": "system", "content": system_prompt})
                messages.append({"role": "user", "content": prompt})

                kwargs = {"model": self.model, "messages": messages, "max_tokens": max_tokens}
                if self.use_temperature:
                    kwargs["temperature"] = temperature
                try:
                    response = self.client.chat.completions.create(**kwargs)
                except Exception as e:
                    if "temperature" not in str(e) or "temperature" not in kwargs:
                        raise
                    # Retry without temperature and skip it for the rest of this session
                    self.use_temperature = False
                    del kwargs["temperature"]
                    response = self.client.chat.completions.create(**kwargs)
                return response.choices[0].message.content
            else:
                raise ValueError(f"Unsupported provider: {self.provider}")
        except Exception as e:
            print(f"Error generating response: {e}")
            return f"Error: {str(e)}"

    def is_available(self) -> bool:
        """Check if the LLM client is properly configured."""
        return self.api_key is not None and len(self.api_key) > 0


### Initialize the LLM
Instantiate the configured LLM client and confirm that it is available.

In [24]:
llm = LLMClient(
    model=os.environ.get(
        "PORTKEY_MODEL",
        "@vertexai/anthropic.claude-sonnet-5-5"
    )
)

print("LLM available:", llm.is_available())

LLM available: True


### Build the Initial Pattern-Generation Prompt
Construct a prompt requesting 16 baseline ALU patterns without any coverage feedback.

In [25]:
initial_prompt = f"""
You are generating an INITIAL baseline stimulus set for an 8-bit ALU.

Natural-language specification:

{spec_text}

Verilog DUT:

{rtl_text}

Generate exactly 16 input test patterns.

Requirements:
- This is the INITIAL baseline before any coverage analysis.
- Do NOT use or assume coverage information.
- Do NOT generate expected outputs.
- Do NOT generate a golden model.
- Generate only input patterns.
- Each test pattern must contain a, b, and op.
- a and b are 8-bit values.
- op is a 4-bit value.
- Use plain integer values in the JSON.
- Include a reasonable variety of operations, normal values,
  boundary values, and a few corner cases.

Return ONLY valid JSON in this exact format:

[
  {{"a": 0, "b": 0, "op": 0}},
  {{"a": 1, "b": 2, "op": 0}}
]
"""

### Generate Initial Test Patterns
Send the baseline prompt to the LLM and display its response.

In [26]:
response = llm.generate(
    initial_prompt,
    system_prompt="You are an expert in Verilog verification and digital logic.",
    temperature=0.7,
    max_tokens=2000
)

print(response)

[
  {"a": 0, "b": 0, "op": 0},
  {"a": 255, "b": 1, "op": 0},
  {"a": 127, "b": 1, "op": 0},
  {"a": 128, "b": 1, "op": 1},
  {"a": 5, "b": 10, "op": 1},
  {"a": 170, "b": 85, "op": 2},
  {"a": 170, "b": 85, "op": 3},
  {"a": 255, "b": 15, "op": 4},
  {"a": 0, "b": 77, "op": 5},
  {"a": 129, "b": 7, "op": 6},
  {"a": 129, "b": 1, "op": 7},
  {"a": 128, "b": 7, "op": 8},
  {"a": 128, "b": 127, "op": 9},
  {"a": 127, "b": 128, "op": 9},
  {"a": 200, "b": 100, "op": 10},
  {"a": 85, "b": 0, "op": 15}
]


### Parse Initial LLM Output
Clean the LLM response and convert the returned JSON into Python test-pattern data.

In [27]:
cleaned = response.strip()

if cleaned.startswith("```"):
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)

initial_patterns = json.loads(cleaned)

print("Number of patterns:", len(initial_patterns))
print(json.dumps(initial_patterns, indent=2))

Number of patterns: 16
[
  {
    "a": 0,
    "b": 0,
    "op": 0
  },
  {
    "a": 255,
    "b": 1,
    "op": 0
  },
  {
    "a": 127,
    "b": 1,
    "op": 0
  },
  {
    "a": 128,
    "b": 1,
    "op": 1
  },
  {
    "a": 5,
    "b": 10,
    "op": 1
  },
  {
    "a": 170,
    "b": 85,
    "op": 2
  },
  {
    "a": 170,
    "b": 85,
    "op": 3
  },
  {
    "a": 255,
    "b": 15,
    "op": 4
  },
  {
    "a": 0,
    "b": 77,
    "op": 5
  },
  {
    "a": 129,
    "b": 7,
    "op": 6
  },
  {
    "a": 129,
    "b": 1,
    "op": 7
  },
  {
    "a": 128,
    "b": 7,
    "op": 8
  },
  {
    "a": 128,
    "b": 127,
    "op": 9
  },
  {
    "a": 127,
    "b": 128,
    "op": 9
  },
  {
    "a": 200,
    "b": 100,
    "op": 10
  },
  {
    "a": 85,
    "b": 0,
    "op": 15
  }
]


### Save Initial Pattern Set
Save the 16 baseline patterns to alu_test_patterns_initial.json.

In [28]:
initial_json_path = OUTPUT_DIR / "alu_test_patterns_initial.json"

with open(initial_json_path, "w") as f:
    json.dump(initial_patterns, f, indent=2)

print("Saved:", initial_json_path)

Saved: alu_coverage_output/alu_test_patterns_initial.json


### Generate Iteration 0 Testbench
Create the initial stimulus-only Verilog testbench and enable VCD waveform dumping.

In [29]:
OUTPUT_DIR = Path("alu_coverage_output")

patterns_path = OUTPUT_DIR / "alu_test_patterns_initial.json"
tb_path = OUTPUT_DIR / "alu_tb_iter0.v"
vcd_path = OUTPUT_DIR / "alu_iter0.vcd"

with open(patterns_path, "r") as f:
    patterns = json.load(f)

tb_lines = [
    "`timescale 1ns/1ps",
    "",
    "module alu_tb_iter0;",
    "",
    "    reg  [7:0] a;",
    "    reg  [7:0] b;",
    "    reg  [3:0] op;",
    "",
    "    wire [7:0] y;",
    "    wire zero;",
    "    wire negative;",
    "    wire carry;",
    "    wire overflow;",
    "",
    "    coverage_guided_alu dut (",
    "        .a(a),",
    "        .b(b),",
    "        .op(op),",
    "        .y(y),",
    "        .zero(zero),",
    "        .negative(negative),",
    "        .carry(carry),",
    "        .overflow(overflow)",
    "    );",
    "",
    "    initial begin",
    f'        $dumpfile("{vcd_path}");',
    '        $dumpvars(0, alu_tb_iter0);',
    "",
]

for i, p in enumerate(patterns, start=1):
    tb_lines += [
        f"        // Test {i}",
        f"        a  = 8'd{p['a']};",
        f"        b  = 8'd{p['b']};",
        f"        op = 4'd{p['op']};",
        "        #10;",
        "",
    ]

tb_lines += [
    '        $display("Baseline stimulus complete.");',
    "        $finish;",
    "    end",
    "",
    "endmodule",
]

tb_text = "\n".join(tb_lines)

tb_path.write_text(tb_text)

print(f"Generated: {tb_path}")
print(f"Number of patterns: {len(patterns)}")
print()
print(tb_text)

Generated: alu_coverage_output/alu_tb_iter0.v
Number of patterns: 16

`timescale 1ns/1ps

module alu_tb_iter0;

    reg  [7:0] a;
    reg  [7:0] b;
    reg  [3:0] op;

    wire [7:0] y;
    wire zero;
    wire negative;
    wire carry;
    wire overflow;

    coverage_guided_alu dut (
        .a(a),
        .b(b),
        .op(op),
        .y(y),
        .zero(zero),
        .negative(negative),
        .carry(carry),
        .overflow(overflow)
    );

    initial begin
        $dumpfile("alu_coverage_output/alu_iter0.vcd");
        $dumpvars(0, alu_tb_iter0);

        // Test 1
        a  = 8'd0;
        b  = 8'd0;
        op = 4'd0;
        #10;

        // Test 2
        a  = 8'd255;
        b  = 8'd1;
        op = 4'd0;
        #10;

        // Test 3
        a  = 8'd127;
        b  = 8'd1;
        op = 4'd0;
        #10;

        // Test 4
        a  = 8'd128;
        b  = 8'd1;
        op = 4'd1;
        #10;

        // Test 5
        a  = 8'd5;
        b  = 8'd10;
        op = 

### Compile and Simulate Iteration 0
Compile the baseline testbench with Icarus Verilog and run the simulation.

In [44]:
compile_cmd = [
    "iverilog",
    "-g2012",
    "-o",
    str(OUTPUT_DIR / "alu_iter0.vvp"),
    "coverage_guided_alu.v",
    str(OUTPUT_DIR / "alu_tb_iter0.v"),
]

compile_result = subprocess.run(
    compile_cmd,
    capture_output=True,
    text=True
)

print("=== COMPILE ===")
print("return code:", compile_result.returncode)

if compile_result.stdout:
    print(compile_result.stdout)

if compile_result.stderr:
    print(compile_result.stderr)

if compile_result.returncode == 0:
    sim_result = subprocess.run(
        ["vvp", str(OUTPUT_DIR / "alu_iter0.vvp")],
        capture_output=True,
        text=True
    )

    print("\n=== SIMULATION ===")
    print(sim_result.stdout)

    if sim_result.stderr:
        print(sim_result.stderr)

=== COMPILE ===
return code: 0

=== SIMULATION ===
VCD info: dumpfile alu_coverage_output/alu_iter0.vcd opened for output.
Baseline stimulus complete.
alu_coverage_output/alu_tb_iter0.v:127: $finish called at 160000 (1ps)



### Clean Iteration 0 VCD
Remove the incompatible VCD comment before running Covered.

In [46]:
raw_vcd_path = OUTPUT_DIR / "alu_iter0.vcd"
clean_vcd_path = OUTPUT_DIR / "alu_iter0_clean.vcd"

lines = raw_vcd_path.read_text().splitlines()

cleaned_lines = [
    line for line in lines
    if not line.lstrip().startswith("$comment Show")
]

clean_vcd_path.write_text("\n".join(cleaned_lines) + "\n")

print("Cleaned VCD saved to:", clean_vcd_path)

Cleaned VCD saved to: alu_coverage_output/alu_iter0_clean.vcd


### Score Iteration 0 Coverage
Run Covered on the baseline waveform and generate the coverage database.

In [48]:
OUTPUT_DIR = Path("alu_coverage_output")

vcd_path = OUTPUT_DIR / "alu_iter0.vcd"
cdd_path = OUTPUT_DIR / "alu_iter0.cdd"
summary_path = OUTPUT_DIR / "coverage_summary_iter0.rpt"
detail_path = OUTPUT_DIR / "coverage_detail_iter0.rpt"
covered = ".tools/covered-build/covered-0.7.10/src/covered"

score_cmd = [
    covered,
    "score",
    "-t", "coverage_guided_alu",
    "-i", "alu_tb_iter0.dut",
    "-v", "coverage_guided_alu.v",
    "-vcd", str(clean_vcd_path),
    "-o", str(cdd_path),
]

score_result = subprocess.run(
    score_cmd,
    capture_output=True,
    text=True
)

print("=== COVERED SCORE ===")
print("return code:", score_result.returncode)

if score_result.stdout:
    print(score_result.stdout)

if score_result.stderr:
    print(score_result.stderr)

=== COVERED SCORE ===
return code: 0

Covered covered-0.7.10 -- Verilog Code Coverage Utility
Written by Trevor Williams  (phase1geo@gmail.com)
Freely distributable under the GPL license

Reading design...
Parsing file 'coverage_guided_alu.v'

Checking for race conditions...

Scoring VCD dumpfile alu_coverage_output/alu_iter0_clean.vcd...

***  Scoring completed successfully!  ***

Dynamic memory allocated:   176350 bytes




### Generate Iteration 0 Summary Report
Save and display the baseline coverage summary.

In [49]:
summary_result = subprocess.run(
    [
        covered,
        "report",
        str(cdd_path),
    ],
    capture_output=True,
    text=True
)

summary_path.write_text(summary_result.stdout)

print("=== COVERAGE SUMMARY ===")
print(summary_result.stdout)
print("Saved:", summary_path)

=== COVERAGE SUMMARY ===

Covered covered-0.7.10 -- Verilog Code Coverage Utility
Written by Trevor Williams  (phase1geo@gmail.com)
Freely distributable under the GPL license

                            :::::::::::::::::::::::::::::::::::::::::::::::::::::
                            ::                                                 ::
                            ::  Covered -- Verilog Coverage Summarized Report  ::
                            ::                                                 ::
                            :::::::::::::::::::::::::::::::::::::::::::::::::::::


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~   GENERAL INFORMATION   ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
* Report generated from CDD file : alu_coverage_output/alu

### Generate Iteration 0 Detailed Report
Save and display detailed uncovered logic information for the baseline.

In [50]:
detail_result = subprocess.run(
    [
        covered,
        "report",
        "-d", "v",
        str(cdd_path),
    ],
    capture_output=True,
    text=True
)

detail_path.write_text(detail_result.stdout)

print("=== COVERAGE DETAIL ===")
print(detail_result.stdout)
print("Saved:", detail_path)

=== COVERAGE DETAIL ===

Covered covered-0.7.10 -- Verilog Code Coverage Utility
Written by Trevor Williams  (phase1geo@gmail.com)
Freely distributable under the GPL license

                             ::::::::::::::::::::::::::::::::::::::::::::::::::
                             ::                                              ::
                             ::  Covered -- Verilog Coverage Verbose Report  ::
                             ::                                              ::
                             ::::::::::::::::::::::::::::::::::::::::::::::::::


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~   GENERAL INFORMATION   ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
* Report generated from CDD file : alu_coverage_output/alu_iter0.cdd


### Build Iteration 1 Coverage Feedback
Describe the remaining baseline coverage gaps for the first refinement prompt.

In [51]:
coverage_feedback = """
Current baseline coverage:
- Line coverage: 100%
- Toggle 0->1 coverage: 100%
- Toggle 1->0 coverage: 97%
- Combinational logic coverage: 63%

Uncovered logic targets include:
- ADD: missing additional operand/sign combinations
- ADD overflow logic: missing some XOR input combinations
- SUB: several arithmetic and overflow combinations remain uncovered
- AND: missing cases where one side is 0
- OR: missing case where both operands produce 0 bits
- XOR: missing equal-bit and mixed-bit combinations
- NOT: missing one input-state case
- SLL: missing one shift/input-state case
- SRL: missing one shift/input-state case
- SRA: missing one shift/input-state case
- Toggle coverage: one 1->0 transition on op is still missing

Generate ONLY additional test patterns that target these uncovered cases.

Do not repeat the existing patterns unless absolutely necessary.
Return only valid JSON.
Each entry must contain:
"a": 0-255
"b": 0-255
"op": 0-15

Generate 8 to 12 new patterns.
"""

### Generate Iteration 1 Patterns
Ask the LLM for additional patterns targeting the uncovered iteration 0 behaviors.

In [52]:
refine_response = llm.generate(
    coverage_feedback,
    system_prompt="You are an expert in coverage-guided Verilog verification.",
    temperature=0.5,
    max_tokens=2000
)

print(refine_response)

```json
[
  {"a": 127, "b": 1,   "op": 0},
  {"a": 128, "b": 1,   "op": 1},
  {"a": 127, "b": 255, "op": 1},
  {"a": 0,   "b": 255, "op": 2},
  {"a": 0,   "b": 0,   "op": 3},
  {"a": 170, "b": 170, "op": 4},
  {"a": 85,  "b": 170, "op": 4},
  {"a": 255, "b": 0,   "op": 5},
  {"a": 1,   "b": 7,   "op": 6},
  {"a": 128, "b": 7,   "op": 7},
  {"a": 128, "b": 1,   "op": 8},
  {"a": 128, "b": 128, "op": 0}
]
```

I assumed an op encoding of 0 ADD, 1 SUB, 2 AND, 3 OR, 4 XOR, 5 NOT, 6 SLL, 7 SRL, 8 SRA, since the opcode map wasn't given. I also assumed the shift amount comes from the low bits of b. If your ALU uses a different mapping, the op values will need adjusting. The last two patterns (op 8 followed by op 0) are meant to produce the missing 1->0 transition on op bit 3.


### Parse and Save Iteration 1 Additions
Extract the new LLM-generated patterns and save them as the iteration 1 additions.

In [54]:
cleaned = refine_response.strip()

cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
cleaned = re.sub(r"\s*```$", "", cleaned)

match = re.search(r"\[\s*\{.*\}\s*\]", cleaned, re.DOTALL)

if not match:
    raise ValueError("Could not find JSON array in refinement response")

iter1_additions = json.loads(match.group(0))

print("New patterns:", len(iter1_additions))
print(json.dumps(iter1_additions, indent=2))

iter1_path = OUTPUT_DIR / "alu_test_patterns_iter1_additions.json"

with open(iter1_path, "w") as f:
    json.dump(iter1_additions, f, indent=2)

print("Saved:", iter1_path)

New patterns: 12
[
  {
    "a": 127,
    "b": 1,
    "op": 0
  },
  {
    "a": 128,
    "b": 1,
    "op": 1
  },
  {
    "a": 127,
    "b": 255,
    "op": 1
  },
  {
    "a": 0,
    "b": 255,
    "op": 2
  },
  {
    "a": 0,
    "b": 0,
    "op": 3
  },
  {
    "a": 170,
    "b": 170,
    "op": 4
  },
  {
    "a": 85,
    "b": 170,
    "op": 4
  },
  {
    "a": 255,
    "b": 0,
    "op": 5
  },
  {
    "a": 1,
    "b": 7,
    "op": 6
  },
  {
    "a": 128,
    "b": 7,
    "op": 7
  },
  {
    "a": 128,
    "b": 1,
    "op": 8
  },
  {
    "a": 128,
    "b": 128,
    "op": 0
  }
]
Saved: alu_coverage_output/alu_test_patterns_iter1_additions.json


### Merge Iteration 1 Patterns
Combine the original 16 patterns with the new additions to create the full iteration 1 set.

In [55]:
import json
from pathlib import Path

OUTPUT_DIR = Path("alu_coverage_output")

initial_path = OUTPUT_DIR / "alu_test_patterns_initial.json"
iter1_add_path = OUTPUT_DIR / "alu_test_patterns_iter1_additions.json"
iter1_all_path = OUTPUT_DIR / "alu_test_patterns_iter1_all.json"

with open(initial_path, "r") as f:
    initial_patterns = json.load(f)

with open(iter1_add_path, "r") as f:
    iter1_additions = json.load(f)

iter1_patterns = initial_patterns + iter1_additions

with open(iter1_all_path, "w") as f:
    json.dump(iter1_patterns, f, indent=2)

print("Initial patterns:", len(initial_patterns))
print("New additions:", len(iter1_additions))
print("Total iter1 patterns:", len(iter1_patterns))
print("Saved:", iter1_all_path)

Initial patterns: 16
New additions: 12
Total iter1 patterns: 28
Saved: alu_coverage_output/alu_test_patterns_iter1_all.json


### Generate Iteration 1 Testbench
Create a stimulus-only Verilog testbench using all iteration 1 patterns.

In [56]:
tb_path = OUTPUT_DIR / "alu_tb_iter1.v"
vcd_path_iter1 = OUTPUT_DIR / "alu_iter1.vcd"

tb_lines = [
    "`timescale 1ns/1ps",
    "",
    "module alu_tb_iter1;",
    "",
    "    reg  [7:0] a;",
    "    reg  [7:0] b;",
    "    reg  [3:0] op;",
    "",
    "    wire [7:0] y;",
    "    wire zero;",
    "    wire negative;",
    "    wire carry;",
    "    wire overflow;",
    "",
    "    coverage_guided_alu dut (",
    "        .a(a),",
    "        .b(b),",
    "        .op(op),",
    "        .y(y),",
    "        .zero(zero),",
    "        .negative(negative),",
    "        .carry(carry),",
    "        .overflow(overflow)",
    "    );",
    "",
    "    initial begin",
    f'        $dumpfile("{vcd_path_iter1}");',
    "        $dumpvars(0, alu_tb_iter1);",
    "",
]

for i, p in enumerate(iter1_patterns, start=1):
    tb_lines += [
        f"        // Test {i}",
        f"        a  = 8'd{p['a']};",
        f"        b  = 8'd{p['b']};",
        f"        op = 4'd{p['op']};",
        "        #10;",
        "",
    ]

tb_lines += [
    '        $display("Iteration 1 stimulus complete.");',
    "        $finish;",
    "    end",
    "",
    "endmodule",
]

tb_text = "\n".join(tb_lines)
tb_path.write_text(tb_text)

print("Generated:", tb_path)
print("Patterns:", len(iter1_patterns))

Generated: alu_coverage_output/alu_tb_iter1.v
Patterns: 28


### Compile and Simulate Iteration 1
Compile and run the iteration 1 testbench with Icarus Verilog.

In [57]:
import subprocess

vvp_path_iter1 = OUTPUT_DIR / "alu_iter1.vvp"

compile_cmd = [
    "iverilog",
    "-g2012",
    "-o",
    str(vvp_path_iter1),
    "coverage_guided_alu.v",
    str(tb_path),
]

compile_result = subprocess.run(
    compile_cmd,
    capture_output=True,
    text=True
)

print("=== COMPILE ===")
print("return code:", compile_result.returncode)

if compile_result.stdout:
    print(compile_result.stdout)

if compile_result.stderr:
    print(compile_result.stderr)

if compile_result.returncode == 0:
    sim_result = subprocess.run(
        ["vvp", str(vvp_path_iter1)],
        capture_output=True,
        text=True
    )

    print("\n=== SIMULATION ===")
    print(sim_result.stdout)

    if sim_result.stderr:
        print(sim_result.stderr)

=== COMPILE ===
return code: 0

=== SIMULATION ===
VCD info: dumpfile alu_coverage_output/alu_iter1.vcd opened for output.
Iteration 1 stimulus complete.
alu_coverage_output/alu_tb_iter1.v:199: $finish called at 280000 (1ps)



### Clean Iteration 1 VCD
Prepare the iteration 1 waveform for Covered by removing the unsupported VCD comment.

In [58]:
clean_vcd_path_iter1 = OUTPUT_DIR / "alu_iter1_clean.vcd"

lines = vcd_path_iter1.read_text().splitlines()

cleaned_lines = [
    line for line in lines
    if not line.lstrip().startswith("$comment Show")
]

clean_vcd_path_iter1.write_text("\n".join(cleaned_lines) + "\n")

print("Cleaned VCD saved to:", clean_vcd_path_iter1)

Cleaned VCD saved to: alu_coverage_output/alu_iter1_clean.vcd


### Score Iteration 1 Coverage
Run Covered scoring on the iteration 1 simulation results.

In [59]:
cdd_path_iter1 = OUTPUT_DIR / "alu_iter1.cdd"

score_cmd = [
    covered,
    "score",
    "-t", "coverage_guided_alu",
    "-i", "alu_tb_iter1.dut",
    "-v", "coverage_guided_alu.v",
    "-vcd", str(clean_vcd_path_iter1),
    "-o", str(cdd_path_iter1),
]

score_result = subprocess.run(
    score_cmd,
    capture_output=True,
    text=True
)

print("return code:", score_result.returncode)
print("STDOUT:")
print(score_result.stdout)
print("STDERR:")
print(score_result.stderr)

print("CDD exists:", cdd_path_iter1.exists())
print("CDD size:", cdd_path_iter1.stat().st_size if cdd_path_iter1.exists() else 0)

return code: 0
STDOUT:

Covered covered-0.7.10 -- Verilog Code Coverage Utility
Written by Trevor Williams  (phase1geo@gmail.com)
Freely distributable under the GPL license

Reading design...
Parsing file 'coverage_guided_alu.v'

Checking for race conditions...

Scoring VCD dumpfile alu_coverage_output/alu_iter1_clean.vcd...

***  Scoring completed successfully!  ***

Dynamic memory allocated:   176350 bytes


STDERR:

CDD exists: True
CDD size: 9755


### Generate Iteration 1 Coverage Reports
Save and display both the summary and detailed iteration 1 coverage reports.

In [61]:
summary_path_iter1 = OUTPUT_DIR / "coverage_summary_iter1.rpt"
detail_path_iter1 = OUTPUT_DIR / "coverage_detail_iter1.rpt"

summary_result = subprocess.run(
    [covered, "report", str(cdd_path_iter1)],
    capture_output=True,
    text=True
)

summary_path_iter1.write_text(summary_result.stdout)

detail_result = subprocess.run(
    [covered, "report", "-d", "v", str(cdd_path_iter1)],
    capture_output=True,
    text=True
)

detail_path_iter1.write_text(detail_result.stdout)

print("=== ITER1 SUMMARY ===")
print(summary_result.stdout)
print("Saved:", summary_path_iter1)

print("\n=== ITER1 DETAIL ===")
print(detail_result.stdout)
print("Saved:", detail_path_iter1)

=== ITER1 SUMMARY ===

Covered covered-0.7.10 -- Verilog Code Coverage Utility
Written by Trevor Williams  (phase1geo@gmail.com)
Freely distributable under the GPL license

                            :::::::::::::::::::::::::::::::::::::::::::::::::::::
                            ::                                                 ::
                            ::  Covered -- Verilog Coverage Summarized Report  ::
                            ::                                                 ::
                            :::::::::::::::::::::::::::::::::::::::::::::::::::::


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~   GENERAL INFORMATION   ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
* Report generated from CDD file : alu_coverage_output/alu_it

### Build Iteration 2 Coverage Feedback
Construct the second refinement prompt from the remaining iteration 1 logic gaps.

In [62]:
coverage_feedback_iter2 = """
Current iteration 1 coverage:
- Line coverage: 100%
- Toggle 0->1 coverage: 100%
- Toggle 1->0 coverage: 100%
- Combinational logic coverage: 71%

Remaining uncovered logic targets:

ADD:
- Addition expression is missing operand combinations 01 and 10.
- ADD overflow XOR (a[7] ^ b[7]) is missing combination 01.
- ADD overflow XOR (y[7] ^ a[7]) is missing combination 11.

SUB:
- Subtraction expression is missing combinations 00, 01, and 10.
- SUB overflow XOR (a[7] ^ b[7]) is missing combination 11.
- SUB overflow XOR (y[7] ^ a[7]) is missing combinations 00 and 11.
- SUB overflow AND expression is missing the -0 case.

AND:
- Missing the -0 case, where the right operand contributes zero while the left side can be nonzero.

XOR:
- Missing combinations 00, 01, and 10.

SLL:
- Missing the expression state corresponding to a zero-valued operand/input case.

SRL:
- Missing the expression state corresponding to a zero-valued operand/input case.

SRA:
- Missing the expression state corresponding to a zero-valued operand/input case.

Generate ONLY additional test patterns targeting these remaining uncovered cases.

Do not repeat existing patterns unless absolutely necessary.

Return only valid JSON.
Each entry must contain:
"a": 0-255
"b": 0-255
"op": 0-15

Generate 8 to 12 new patterns.
"""

### Generate Iteration 2 Patterns
Ask the LLM for additional patterns targeting the remaining uncovered logic states.

In [63]:
refine_response_iter2 = llm.generate(
    coverage_feedback_iter2,
    system_prompt="You are an expert in coverage-guided Verilog verification.",
    temperature=0.5,
    max_tokens=2000
)

print(refine_response_iter2)

```json
[
  {"a": 144, "b": 5, "op": 0},
  {"a": 5, "b": 144, "op": 0},
  {"a": 0, "b": 5, "op": 1},
  {"a": 5, "b": 0, "op": 1},
  {"a": 144, "b": 133, "op": 1},
  {"a": 16, "b": 5, "op": 1},
  {"a": 255, "b": 0, "op": 2},
  {"a": 0, "b": 0, "op": 4},
  {"a": 255, "b": 0, "op": 4},
  {"a": 0, "b": 3, "op": 5},
  {"a": 0, "b": 3, "op": 6},
  {"a": 0, "b": 3, "op": 7}
]
```


### Parse Iteration 2 LLM Output
Extract the new iteration 2 patterns from the LLM response.

In [65]:
cleaned = refine_response_iter2.strip()

cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
cleaned = re.sub(r"\s*```$", "", cleaned)

match = re.search(r"\[\s*\{.*\}\s*\]", cleaned, re.DOTALL)

if not match:
    raise ValueError("Could not find JSON array in iter2 refinement response")

iter2_additions = json.loads(match.group(0))

print("Iter2 new patterns:", len(iter2_additions))
print(json.dumps(iter2_additions, indent=2))

Iter2 new patterns: 12
[
  {
    "a": 144,
    "b": 5,
    "op": 0
  },
  {
    "a": 5,
    "b": 144,
    "op": 0
  },
  {
    "a": 0,
    "b": 5,
    "op": 1
  },
  {
    "a": 5,
    "b": 0,
    "op": 1
  },
  {
    "a": 144,
    "b": 133,
    "op": 1
  },
  {
    "a": 16,
    "b": 5,
    "op": 1
  },
  {
    "a": 255,
    "b": 0,
    "op": 2
  },
  {
    "a": 0,
    "b": 0,
    "op": 4
  },
  {
    "a": 255,
    "b": 0,
    "op": 4
  },
  {
    "a": 0,
    "b": 3,
    "op": 5
  },
  {
    "a": 0,
    "b": 3,
    "op": 6
  },
  {
    "a": 0,
    "b": 3,
    "op": 7
  }
]


### Save Iteration 2 Additions
Write the newly generated iteration 2 patterns to JSON.

In [66]:
iter2_path = OUTPUT_DIR / "alu_test_patterns_iter2_additions.json"

with open(iter2_path, "w") as f:
    json.dump(iter2_additions, f, indent=2)

print("Saved:", iter2_path)

Saved: alu_coverage_output/alu_test_patterns_iter2_additions.json


### Merge Iteration 2 Patterns
Append the iteration 2 additions to the previous set, producing 40 total patterns.

In [67]:
iter2_add_path = OUTPUT_DIR / "alu_test_patterns_iter2_additions.json"
iter1_all_path = OUTPUT_DIR / "alu_test_patterns_iter1_all.json"
iter2_all_path = OUTPUT_DIR / "alu_test_patterns_iter2_all.json"

with open(iter1_all_path, "r") as f:
    iter1_patterns = json.load(f)

with open(iter2_add_path, "r") as f:
    iter2_additions = json.load(f)

iter2_patterns = iter1_patterns + iter2_additions

with open(iter2_all_path, "w") as f:
    json.dump(iter2_patterns, f, indent=2)

print("Iter1 total:", len(iter1_patterns))
print("Iter2 additions:", len(iter2_additions))
print("Iter2 total:", len(iter2_patterns))
print("Saved:", iter2_all_path)

Iter1 total: 28
Iter2 additions: 12
Iter2 total: 40
Saved: alu_coverage_output/alu_test_patterns_iter2_all.json


### Generate Iteration 2 Testbench
Create the stimulus-only testbench for the 40-pattern iteration 2 set.

In [68]:
tb_path = OUTPUT_DIR / "alu_tb_iter2.v"
vcd_path_iter2 = OUTPUT_DIR / "alu_iter2.vcd"

tb_lines = [
    "`timescale 1ns/1ps",
    "",
    "module alu_tb_iter2;",
    "",
    "    reg  [7:0] a;",
    "    reg  [7:0] b;",
    "    reg  [3:0] op;",
    "",
    "    wire [7:0] y;",
    "    wire zero;",
    "    wire negative;",
    "    wire carry;",
    "    wire overflow;",
    "",
    "    coverage_guided_alu dut (",
    "        .a(a),",
    "        .b(b),",
    "        .op(op),",
    "        .y(y),",
    "        .zero(zero),",
    "        .negative(negative),",
    "        .carry(carry),",
    "        .overflow(overflow)",
    "    );",
    "",
    "    initial begin",
    f'        $dumpfile("{vcd_path_iter2}");',
    "        $dumpvars(0, alu_tb_iter2);",
    "",
]

for i, p in enumerate(iter2_patterns, start=1):
    tb_lines += [
        f"        // Test {i}",
        f"        a  = 8'd{p['a']};",
        f"        b  = 8'd{p['b']};",
        f"        op = 4'd{p['op']};",
        "        #10;",
        "",
    ]

tb_lines += [
    '        $display("Iteration 2 stimulus complete.");',
    "        $finish;",
    "    end",
    "",
    "endmodule",
]

tb_path.write_text("\n".join(tb_lines))

print("Generated:", tb_path)
print("Patterns:", len(iter2_patterns))

Generated: alu_coverage_output/alu_tb_iter2.v
Patterns: 40


### Compile and Simulate Iteration 2
Compile and execute the iteration 2 Verilog testbench.

In [69]:
vvp_path_iter2 = OUTPUT_DIR / "alu_iter2.vvp"

compile_cmd = [
    "iverilog",
    "-g2012",
    "-o",
    str(vvp_path_iter2),
    "coverage_guided_alu.v",
    str(tb_path),
]

compile_result = subprocess.run(
    compile_cmd,
    capture_output=True,
    text=True
)

print("=== COMPILE ===")
print("return code:", compile_result.returncode)
print(compile_result.stdout)
print(compile_result.stderr)

if compile_result.returncode == 0:
    sim_result = subprocess.run(
        ["vvp", str(vvp_path_iter2)],
        capture_output=True,
        text=True
    )

    print("=== SIMULATION ===")
    print(sim_result.stdout)
    print(sim_result.stderr)

=== COMPILE ===
return code: 0


=== SIMULATION ===
VCD info: dumpfile alu_coverage_output/alu_iter2.vcd opened for output.
Iteration 2 stimulus complete.
alu_coverage_output/alu_tb_iter2.v:271: $finish called at 400000 (1ps)




### Clean Iteration 2 VCD
Prepare the iteration 2 VCD waveform for Covered analysis.

In [70]:
clean_vcd_path_iter2 = OUTPUT_DIR / "alu_iter2_clean.vcd"

lines = vcd_path_iter2.read_text().splitlines()

cleaned_lines = [
    line for line in lines
    if not line.lstrip().startswith("$comment Show")
]

clean_vcd_path_iter2.write_text("\n".join(cleaned_lines) + "\n")

print("Cleaned VCD:", clean_vcd_path_iter2)

Cleaned VCD: alu_coverage_output/alu_iter2_clean.vcd


### Score Iteration 2 Coverage
Run Covered scoring using the iteration 2 waveform.

In [71]:
cdd_path_iter2 = OUTPUT_DIR / "alu_iter2.cdd"

score_cmd = [
    covered,
    "score",
    "-t", "coverage_guided_alu",
    "-i", "alu_tb_iter2.dut",
    "-v", "coverage_guided_alu.v",
    "-vcd", str(clean_vcd_path_iter2),
    "-o", str(cdd_path_iter2),
]

score_result = subprocess.run(
    score_cmd,
    capture_output=True,
    text=True
)

print("return code:", score_result.returncode)
print(score_result.stdout)
print(score_result.stderr)

return code: 0

Covered covered-0.7.10 -- Verilog Code Coverage Utility
Written by Trevor Williams  (phase1geo@gmail.com)
Freely distributable under the GPL license

Reading design...
Parsing file 'coverage_guided_alu.v'

Checking for race conditions...

Scoring VCD dumpfile alu_coverage_output/alu_iter2_clean.vcd...

***  Scoring completed successfully!  ***

Dynamic memory allocated:   176350 bytes





### Generate Iteration 2 Coverage Reports
Save and display the iteration 2 summary and detailed coverage results.

In [72]:
summary_path_iter2 = OUTPUT_DIR / "coverage_summary_iter2.rpt"
detail_path_iter2 = OUTPUT_DIR / "coverage_detail_iter2.rpt"

summary_result_iter2 = subprocess.run(
    [covered, "report", str(cdd_path_iter2)],
    capture_output=True,
    text=True
)

summary_path_iter2.write_text(summary_result_iter2.stdout)

detail_result_iter2 = subprocess.run(
    [covered, "report", "-d", "v", str(cdd_path_iter2)],
    capture_output=True,
    text=True
)

detail_path_iter2.write_text(detail_result_iter2.stdout)

print("=== ITER2 SUMMARY ===")
print(summary_result_iter2.stdout)
print("Saved:", summary_path_iter2)

print("\n=== ITER2 DETAIL ===")
print(detail_result_iter2.stdout)
print("Saved:", detail_path_iter2)

=== ITER2 SUMMARY ===

Covered covered-0.7.10 -- Verilog Code Coverage Utility
Written by Trevor Williams  (phase1geo@gmail.com)
Freely distributable under the GPL license

                            :::::::::::::::::::::::::::::::::::::::::::::::::::::
                            ::                                                 ::
                            ::  Covered -- Verilog Coverage Summarized Report  ::
                            ::                                                 ::
                            :::::::::::::::::::::::::::::::::::::::::::::::::::::


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~   GENERAL INFORMATION   ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
* Report generated from CDD file : alu_coverage_output/alu_it

### Build Iteration 3 Refinement Prompt
Include the current 40 patterns, coverage metrics, and six remaining uncovered logic targets in the final refinement prompt.

In [73]:
import json

# Load the complete current pattern set
iter2_all_path = OUTPUT_DIR / "alu_test_patterns_iter2_all.json"

with open(iter2_all_path, "r") as f:
    iter2_patterns = json.load(f)

coverage_feedback_iter3 = f"""
We are performing coverage-guided refinement of an 8-bit ALU testbench.

ALU operations:
op=0: ADD, y = a + b
op=1: SUB, y = a - b
op=2: AND, y = a & b
op=3: OR, y = a | b
op=4: XOR, y = a ^ b
op=5: NOT, y = ~a
op=6: SLL, y = a << b[2:0]
op=7: SRL, y = a >> b[2:0]
op=8: SRA, y = signed(a) >>> b[2:0]
op=9: signed SLT
op=10-15: default behavior

Current coverage after iteration 2:
- Line coverage: 100%
- Toggle 0->1 coverage: 100%
- Toggle 1->0 coverage: 100%
- Combinational logic coverage: 57/63 = 90%

Covered reports exactly 6 remaining missed logic combinations:

1. ADD, op=0:
   Expression {{carry, y}} = {{1'b0,a}} + {{1'b0,b}}
   Missing combinations: 01 and 10.

2. SUB, op=1:
   Expression y = a - b
   Missing combination: 00.

3. SUB overflow logic, op=1:
   Expression (y[7] ^ a[7])
   Missing combination: 11.

4. XOR, op=4:
   Expression y = a ^ b
   Missing combination: 01.

5. SRA, op=8:
   Expression signed(a) >>> b[2:0]
   Missing expression state E=0.

Current pattern set:
{json.dumps(iter2_patterns, indent=2)}

Generate ONLY additional targeted patterns intended to hit these remaining uncovered combinations.
Do not delete existing patterns.
Avoid duplicating existing patterns unless absolutely necessary.

Return ONLY valid JSON in exactly this format:

{{
  "patterns": [
    {{
      "a": 0,
      "b": 0,
      "op": 0,
      "target": "short explanation of which uncovered behavior this pattern targets"
    }}
  ]
}}

Generate approximately 5-10 new patterns.
Every pattern MUST include a short target explanation.
"""

print(coverage_feedback_iter3)


We are performing coverage-guided refinement of an 8-bit ALU testbench.

ALU operations:
op=0: ADD, y = a + b
op=1: SUB, y = a - b
op=2: AND, y = a & b
op=3: OR, y = a | b
op=4: XOR, y = a ^ b
op=5: NOT, y = ~a
op=6: SLL, y = a << b[2:0]
op=7: SRL, y = a >> b[2:0]
op=8: SRA, y = signed(a) >>> b[2:0]
op=9: signed SLT
op=10-15: default behavior

Current coverage after iteration 2:
- Line coverage: 100%
- Toggle 0->1 coverage: 100%
- Toggle 1->0 coverage: 100%
- Combinational logic coverage: 57/63 = 90%

Covered reports exactly 6 remaining missed logic combinations:

1. ADD, op=0:
   Expression {carry, y} = {1'b0,a} + {1'b0,b}
   Missing combinations: 01 and 10.

2. SUB, op=1:
   Expression y = a - b
   Missing combination: 00.

3. SUB overflow logic, op=1:
   Expression (y[7] ^ a[7])
   Missing combination: 11.

4. XOR, op=4:
   Expression y = a ^ b
   Missing combination: 01.

5. SRA, op=8:
   Expression signed(a) >>> b[2:0]
   Missing expression state E=0.

Current pattern set:
[
  {


### Generate Iteration 3 Targeted Patterns
Ask the LLM for final targeted patterns with an explanation of the coverage target for each one.

In [74]:
refine_response_iter3 = llm.generate(
    coverage_feedback_iter3,
    system_prompt="You are an expert in coverage-guided Verilog verification.",
    temperature=0.4,
    max_tokens=2500
)

print(refine_response_iter3)

{
  "patterns": [
    {"a": 1, "b": 128, "op": 0, "target": "ADD: a[7]=0,b[7]=1 operand-MSB combo 01, no carry"},
    {"a": 100, "b": 50, "op": 0, "target": "ADD: no carry with y[7]=1 (carry=0, y7=1)"},
    {"a": 200, "b": 100, "op": 0, "target": "ADD: carry=1 with y[7]=0 (300 wraps to 44)"},
    {"a": 100, "b": 50, "op": 1, "target": "SUB: no borrow, positive result, 00 combination"},
    {"a": 77, "b": 77, "op": 1, "target": "SUB: zero result, a==b, 00 combination"},
    {"a": 200, "b": 10, "op": 1, "target": "SUB overflow term (y[7]^a[7]): y7=1 and a7=1 (190 from 200)"},
    {"a": 0, "b": 255, "op": 4, "target": "XOR: a bit=0, b bit=1 combination 01"},
    {"a": 127, "b": 7, "op": 8, "target": "SRA: positive a shifted by 7 gives 0, expression state E=0"},
    {"a": 0, "b": 3, "op": 8, "target": "SRA: zero input gives zero result, E=0"}
  ]
}


### Parse Iteration 3 LLM Output
Convert the structured iteration 3 response into Python data.

In [75]:
import re
import json

cleaned = refine_response_iter3.strip()

cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
cleaned = re.sub(r"\s*```$", "", cleaned)

iter3_result = json.loads(cleaned)

iter3_entries = iter3_result["patterns"]

print("Iter3 additions:", len(iter3_entries))
print(json.dumps(iter3_entries, indent=2))

Iter3 additions: 9
[
  {
    "a": 1,
    "b": 128,
    "op": 0,
    "target": "ADD: a[7]=0,b[7]=1 operand-MSB combo 01, no carry"
  },
  {
    "a": 100,
    "b": 50,
    "op": 0,
    "target": "ADD: no carry with y[7]=1 (carry=0, y7=1)"
  },
  {
    "a": 200,
    "b": 100,
    "op": 0,
    "target": "ADD: carry=1 with y[7]=0 (300 wraps to 44)"
  },
  {
    "a": 100,
    "b": 50,
    "op": 1,
    "target": "SUB: no borrow, positive result, 00 combination"
  },
  {
    "a": 77,
    "b": 77,
    "op": 1,
    "target": "SUB: zero result, a==b, 00 combination"
  },
  {
    "a": 200,
    "b": 10,
    "op": 1,
    "target": "SUB overflow term (y[7]^a[7]): y7=1 and a7=1 (190 from 200)"
  },
  {
    "a": 0,
    "b": 255,
    "op": 4,
    "target": "XOR: a bit=0, b bit=1 combination 01"
  },
  {
    "a": 127,
    "b": 7,
    "op": 8,
    "target": "SRA: positive a shifted by 7 gives 0, expression state E=0"
  },
  {
    "a": 0,
    "b": 3,
    "op": 8,
    "target": "SRA: zero input gives zero r

### Save Iteration 3 Target Mappings
Preserve the new patterns together with their intended uncovered-behavior descriptions.

In [76]:
iter3_mapping_path = OUTPUT_DIR / "alu_test_patterns_iter3_with_targets.json"

with open(iter3_mapping_path, "w") as f:
    json.dump(iter3_result, f, indent=2)

print("Saved:", iter3_mapping_path)

Saved: alu_coverage_output/alu_test_patterns_iter3_with_targets.json


### Extract Iteration 3 Stimulus Patterns
Remove the explanatory fields and save plain a, b, and op inputs for simulation.

In [77]:
iter3_additions = [
    {
        "a": entry["a"],
        "b": entry["b"],
        "op": entry["op"]
    }
    for entry in iter3_entries
]

iter3_add_path = OUTPUT_DIR / "alu_test_patterns_iter3_additions.json"

with open(iter3_add_path, "w") as f:
    json.dump(iter3_additions, f, indent=2)

print("Saved:", iter3_add_path)
print(json.dumps(iter3_additions, indent=2))

Saved: alu_coverage_output/alu_test_patterns_iter3_additions.json
[
  {
    "a": 1,
    "b": 128,
    "op": 0
  },
  {
    "a": 100,
    "b": 50,
    "op": 0
  },
  {
    "a": 200,
    "b": 100,
    "op": 0
  },
  {
    "a": 100,
    "b": 50,
    "op": 1
  },
  {
    "a": 77,
    "b": 77,
    "op": 1
  },
  {
    "a": 200,
    "b": 10,
    "op": 1
  },
  {
    "a": 0,
    "b": 255,
    "op": 4
  },
  {
    "a": 127,
    "b": 7,
    "op": 8
  },
  {
    "a": 0,
    "b": 3,
    "op": 8
  }
]


### Reload Iteration 3 Pattern Additions
Reload the saved target mappings and recreate the plain stimulus-only additions.

In [78]:
with open(OUTPUT_DIR / "alu_test_patterns_iter3_with_targets.json", "r") as f:
    iter3_result = json.load(f)

iter3_entries = iter3_result["patterns"]

iter3_additions = [
    {
        "a": p["a"],
        "b": p["b"],
        "op": p["op"]
    }
    for p in iter3_entries
]

iter3_add_path = OUTPUT_DIR / "alu_test_patterns_iter3_additions.json"

with open(iter3_add_path, "w") as f:
    json.dump(iter3_additions, f, indent=2)

print("Iter3 additions:", len(iter3_additions))
print("Saved:", iter3_add_path)

Iter3 additions: 9
Saved: alu_coverage_output/alu_test_patterns_iter3_additions.json


### Merge Iteration 3 Patterns
Combine the iteration 2 set with the nine final additions to produce 49 total patterns.

In [79]:
iter2_all_path = OUTPUT_DIR / "alu_test_patterns_iter2_all.json"
iter3_all_path = OUTPUT_DIR / "alu_test_patterns_iter3_all.json"

with open(iter2_all_path, "r") as f:
    iter2_patterns = json.load(f)

iter3_patterns = iter2_patterns + iter3_additions

with open(iter3_all_path, "w") as f:
    json.dump(iter3_patterns, f, indent=2)

print("Iter2 total:", len(iter2_patterns))
print("Iter3 additions:", len(iter3_additions))
print("Iter3 total:", len(iter3_patterns))
print("Saved:", iter3_all_path)

Iter2 total: 40
Iter3 additions: 9
Iter3 total: 49
Saved: alu_coverage_output/alu_test_patterns_iter3_all.json


### Generate Iteration 3 Testbench
Create the final coverage-refinement stimulus testbench using all 49 patterns.

In [80]:
# 1) Generate alu_tb_iter3.v

tb_path_iter3 = OUTPUT_DIR / "alu_tb_iter3.v"
vcd_path_iter3 = OUTPUT_DIR / "alu_iter3.vcd"

tb_lines = [
    "`timescale 1ns/1ps",
    "",
    "module alu_tb_iter3;",
    "",
    "    reg  [7:0] a;",
    "    reg  [7:0] b;",
    "    reg  [3:0] op;",
    "",
    "    wire [7:0] y;",
    "    wire zero;",
    "    wire negative;",
    "    wire carry;",
    "    wire overflow;",
    "",
    "    coverage_guided_alu dut (",
    "        .a(a),",
    "        .b(b),",
    "        .op(op),",
    "        .y(y),",
    "        .zero(zero),",
    "        .negative(negative),",
    "        .carry(carry),",
    "        .overflow(overflow)",
    "    );",
    "",
    "    initial begin",
    f'        $dumpfile("{vcd_path_iter3}");',
    "        $dumpvars(0, alu_tb_iter3);",
    "",
]

for i, p in enumerate(iter3_patterns, start=1):
    tb_lines += [
        f"        // Test {i}",
        f"        a  = 8'd{p['a']};",
        f"        b  = 8'd{p['b']};",
        f"        op = 4'd{p['op']};",
        "        #10;",
        "",
    ]

tb_lines += [
    '        $display("Iteration 3 stimulus complete.");',
    "        $finish;",
    "    end",
    "",
    "endmodule",
]

tb_path_iter3.write_text("\n".join(tb_lines))

print("Generated:", tb_path_iter3)
print("Patterns:", len(iter3_patterns))

Generated: alu_coverage_output/alu_tb_iter3.v
Patterns: 49


### Compile and Simulate Iteration 3
Compile and run the final coverage-refinement testbench.

In [81]:
# 2) Compile + simulate

vvp_path_iter3 = OUTPUT_DIR / "alu_iter3.vvp"

compile_cmd = [
    "iverilog",
    "-g2012",
    "-o", str(vvp_path_iter3),
    "coverage_guided_alu.v",
    str(tb_path_iter3),
]

compile_result = subprocess.run(
    compile_cmd,
    capture_output=True,
    text=True
)

print("=== COMPILE ===")
print("return code:", compile_result.returncode)

if compile_result.stdout:
    print(compile_result.stdout)

if compile_result.stderr:
    print(compile_result.stderr)

if compile_result.returncode == 0:
    sim_result = subprocess.run(
        ["vvp", str(vvp_path_iter3)],
        capture_output=True,
        text=True
    )

    print("\n=== SIMULATION ===")
    print(sim_result.stdout)

    if sim_result.stderr:
        print(sim_result.stderr)

=== COMPILE ===
return code: 0

=== SIMULATION ===
VCD info: dumpfile alu_coverage_output/alu_iter3.vcd opened for output.
Iteration 3 stimulus complete.
alu_coverage_output/alu_tb_iter3.v:325: $finish called at 490000 (1ps)



### Clean Iteration 3 VCD
Prepare the iteration 3 waveform for Covered analysis.

In [82]:
# 3) Clean the VCD for Covered

clean_vcd_path_iter3 = OUTPUT_DIR / "alu_iter3_clean.vcd"

lines = vcd_path_iter3.read_text().splitlines()

cleaned_lines = [
    line for line in lines
    if not line.lstrip().startswith("$comment Show")
]

clean_vcd_path_iter3.write_text(
    "\n".join(cleaned_lines) + "\n"
)

print("Cleaned VCD:", clean_vcd_path_iter3)

Cleaned VCD: alu_coverage_output/alu_iter3_clean.vcd


### Score Iteration 3 Coverage
Run Covered scoring on the final refinement simulation.

In [83]:
# 4) Run Covered scoring

cdd_path_iter3 = OUTPUT_DIR / "alu_iter3.cdd"

score_cmd = [
    covered,
    "score",
    "-t", "coverage_guided_alu",
    "-i", "alu_tb_iter3.dut",
    "-v", "coverage_guided_alu.v",
    "-vcd", str(clean_vcd_path_iter3),
    "-o", str(cdd_path_iter3),
]

score_result = subprocess.run(
    score_cmd,
    capture_output=True,
    text=True
)

print("=== COVERED SCORE ===")
print("return code:", score_result.returncode)
print(score_result.stdout)

if score_result.stderr:
    print(score_result.stderr)

print("CDD exists:", cdd_path_iter3.exists())

=== COVERED SCORE ===
return code: 0

Covered covered-0.7.10 -- Verilog Code Coverage Utility
Written by Trevor Williams  (phase1geo@gmail.com)
Freely distributable under the GPL license

Reading design...
Parsing file 'coverage_guided_alu.v'

Checking for race conditions...

Scoring VCD dumpfile alu_coverage_output/alu_iter3_clean.vcd...

***  Scoring completed successfully!  ***

Dynamic memory allocated:   176350 bytes


CDD exists: True


### Generate Iteration 3 Coverage Reports
Save and display the final coverage summary and detailed report.

In [84]:
# 5) Generate iter3 coverage reports

summary_path_iter3 = OUTPUT_DIR / "coverage_summary_iter3.rpt"
detail_path_iter3 = OUTPUT_DIR / "coverage_detail_iter3.rpt"

summary_result_iter3 = subprocess.run(
    [covered, "report", str(cdd_path_iter3)],
    capture_output=True,
    text=True
)

summary_path_iter3.write_text(summary_result_iter3.stdout)

detail_result_iter3 = subprocess.run(
    [covered, "report", "-d", "v", str(cdd_path_iter3)],
    capture_output=True,
    text=True
)

detail_path_iter3.write_text(detail_result_iter3.stdout)

print("=== ITER3 SUMMARY ===")
print(summary_result_iter3.stdout)
print("Saved:", summary_path_iter3)

print("\n=== ITER3 DETAIL ===")
print(detail_result_iter3.stdout)
print("Saved:", detail_path_iter3)

=== ITER3 SUMMARY ===

Covered covered-0.7.10 -- Verilog Code Coverage Utility
Written by Trevor Williams  (phase1geo@gmail.com)
Freely distributable under the GPL license

                            :::::::::::::::::::::::::::::::::::::::::::::::::::::
                            ::                                                 ::
                            ::  Covered -- Verilog Coverage Summarized Report  ::
                            ::                                                 ::
                            :::::::::::::::::::::::::::::::::::::::::::::::::::::


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~   GENERAL INFORMATION   ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
* Report generated from CDD file : alu_coverage_output/alu_it

### Save Coverage-Optimized Pattern Set
Save the final 49-pattern set as alu_test_patterns_coverage_optimized.json.

In [85]:
final_patterns_path = OUTPUT_DIR / "alu_test_patterns_coverage_optimized.json"

with open(OUTPUT_DIR / "alu_test_patterns_iter3_all.json", "r") as f:
    optimized_patterns = json.load(f)

with open(final_patterns_path, "w") as f:
    json.dump(optimized_patterns, f, indent=2)

print("Final optimized patterns:", len(optimized_patterns))
print("Saved:", final_patterns_path)

Final optimized patterns: 49
Saved: alu_coverage_output/alu_test_patterns_coverage_optimized.json


### Save Coverage History
Record pattern counts and coverage percentages for iterations 0 through 3.

In [86]:
coverage_history = [
    {
        "iteration": 0,
        "patterns": 16,
        "line": 100,
        "toggle_0_to_1": 100,
        "toggle_1_to_0": 97,
        "logic": 63
    },
    {
        "iteration": 1,
        "patterns": 28,
        "line": 100,
        "toggle_0_to_1": 100,
        "toggle_1_to_0": 100,
        "logic": 71
    },
    {
        "iteration": 2,
        "patterns": 40,
        "line": 100,
        "toggle_0_to_1": 100,
        "toggle_1_to_0": 100,
        "logic": 90
    },
    {
        "iteration": 3,
        "patterns": 49,
        "line": 100,
        "toggle_0_to_1": 100,
        "toggle_1_to_0": 100,
        "logic": 95
    }
]

coverage_history_path = OUTPUT_DIR / "coverage_history.json"

with open(coverage_history_path, "w") as f:
    json.dump(coverage_history, f, indent=2)

print("Saved:", coverage_history_path)

Saved: alu_coverage_output/coverage_history.json


### Define the Python Golden Model
Implement the ALU specification independently in Python to calculate expected results and flags.

In [87]:
def to_signed8(x):
    return x - 256 if x >= 128 else x

def golden_alu(a, b, op):
    a &= 0xFF
    b &= 0xFF
    op &= 0xF

    y = 0
    carry = 0
    overflow = 0

    if op == 0:  # ADD
        total = a + b
        y = total & 0xFF
        carry = 1 if total > 0xFF else 0

        sa = (a >> 7) & 1
        sb = (b >> 7) & 1
        sy = (y >> 7) & 1
        overflow = 1 if (sa == sb and sy != sa) else 0

    elif op == 1:  # SUB
        y = (a - b) & 0xFF
        carry = 1 if a >= b else 0

        sa = (a >> 7) & 1
        sb = (b >> 7) & 1
        sy = (y >> 7) & 1
        overflow = 1 if (sa != sb and sy != sa) else 0

    elif op == 2:
        y = a & b

    elif op == 3:
        y = a | b

    elif op == 4:
        y = a ^ b

    elif op == 5:
        y = (~a) & 0xFF

    elif op == 6:
        y = (a << (b & 0x7)) & 0xFF

    elif op == 7:
        y = (a >> (b & 0x7)) & 0xFF

    elif op == 8:
        shift = b & 0x7
        y = (to_signed8(a) >> shift) & 0xFF

    elif op == 9:
        y = 1 if to_signed8(a) < to_signed8(b) else 0

    zero = 1 if y == 0 else 0
    negative = (y >> 7) & 1

    return {
        "y": y,
        "zero": zero,
        "negative": negative,
        "carry": carry,
        "overflow": overflow
    }

### Generate Golden Expected Outputs
Apply the golden model to the baseline and optimized patterns and save the labeled JSON files.

In [88]:
with open(OUTPUT_DIR / "alu_test_patterns_initial.json", "r") as f:
    baseline_patterns = json.load(f)

with open(OUTPUT_DIR / "alu_test_patterns_coverage_optimized.json", "r") as f:
    optimized_patterns = json.load(f)

def label_patterns(patterns):
    labeled = []

    for p in patterns:
        expected = golden_alu(p["a"], p["b"], p["op"])

        labeled.append({
            "a": p["a"],
            "b": p["b"],
            "op": p["op"],
            **expected
        })

    return labeled

baseline_with_golden = label_patterns(baseline_patterns)
optimized_with_golden = label_patterns(optimized_patterns)

with open(OUTPUT_DIR / "alu_test_patterns_initial_with_golden.json", "w") as f:
    json.dump(baseline_with_golden, f, indent=2)

with open(OUTPUT_DIR / "alu_test_patterns_with_golden.json", "w") as f:
    json.dump(optimized_with_golden, f, indent=2)

print("Baseline:", len(baseline_with_golden))
print("Optimized:", len(optimized_with_golden))

Baseline: 16
Optimized: 49


### Create an Intentional Buggy ALU
Create a modified DUT where arithmetic right shift incorrectly behaves as logical right shift.

In [89]:
correct_rtl_path = Path("coverage_guided_alu.v")
buggy_rtl_path = OUTPUT_DIR / "coverage_guided_alu_buggy.v"

rtl_text = correct_rtl_path.read_text()

old_line = "y = $signed(a) >>> b[2:0];"
new_line = "y = a >> b[2:0];"

if old_line not in rtl_text:
    raise ValueError("Could not find the SRA line in coverage_guided_alu.v")

buggy_text = rtl_text.replace(old_line, new_line, 1)

buggy_rtl_path.write_text(buggy_text)

print("Created:", buggy_rtl_path)
print("Intentional bug: SRA now behaves like SRL")

Created: alu_coverage_output/coverage_guided_alu_buggy.v
Intentional bug: SRA now behaves like SRL


### Generate Self-Checking Bug Testbenches
Build self-checking testbenches for the baseline and optimized pattern sets using golden expected outputs.

In [90]:
def make_self_checking_tb(patterns_with_golden, module_name, tb_path, dut_filename):
    lines = [
        "`timescale 1ns/1ps",
        "",
        f"module {module_name};",
        "",
        "    reg  [7:0] a;",
        "    reg  [7:0] b;",
        "    reg  [3:0] op;",
        "",
        "    wire [7:0] y;",
        "    wire zero;",
        "    wire negative;",
        "    wire carry;",
        "    wire overflow;",
        "",
        "    integer failures = 0;",
        "",
        "    coverage_guided_alu dut (",
        "        .a(a),",
        "        .b(b),",
        "        .op(op),",
        "        .y(y),",
        "        .zero(zero),",
        "        .negative(negative),",
        "        .carry(carry),",
        "        .overflow(overflow)",
        "    );",
        "",
        "    initial begin",
    ]

    for i, p in enumerate(patterns_with_golden, start=1):
        lines += [
            f"        // Test {i}",
            f"        a  = 8'd{p['a']};",
            f"        b  = 8'd{p['b']};",
            f"        op = 4'd{p['op']};",
            "        #10;",
            "",
            "        if (",
            f"            y !== 8'd{p['y']} ||",
            f"            zero !== 1'b{p['zero']} ||",
            f"            negative !== 1'b{p['negative']} ||",
            f"            carry !== 1'b{p['carry']} ||",
            f"            overflow !== 1'b{p['overflow']}",
            "        ) begin",
            f'            $display("FAIL test {i}: a=%0d b=%0d op=%0d | got y=%0d z=%0d n=%0d c=%0d v=%0d | expected y={p["y"]} z={p["zero"]} n={p["negative"]} c={p["carry"]} v={p["overflow"]}",',
            "                     a, b, op, y, zero, negative, carry, overflow);",
            "            failures = failures + 1;",
            "        end",
            "",
        ]

    lines += [
        '        $display("TOTAL FAILURES: %0d", failures);',
        "        $finish;",
        "    end",
        "",
        "endmodule",
    ]

    Path(tb_path).write_text("\n".join(lines))


# Paths
baseline_bug_tb = OUTPUT_DIR / "alu_tb_bug_baseline.v"
optimized_bug_tb = OUTPUT_DIR / "alu_tb_bug_optimized.v"

make_self_checking_tb(
    baseline_with_golden,
    "alu_tb_bug_baseline",
    baseline_bug_tb,
    buggy_rtl_path
)

make_self_checking_tb(
    optimized_with_golden,
    "alu_tb_bug_optimized",
    optimized_bug_tb,
    buggy_rtl_path
)

print("Generated:")
print(baseline_bug_tb)
print(optimized_bug_tb)

Generated:
alu_coverage_output/alu_tb_bug_baseline.v
alu_coverage_output/alu_tb_bug_optimized.v


### Run the Bug-Detection Comparison
Compile and simulate both testbenches against the buggy ALU and compare detected failures.

In [91]:
def compile_and_run_bug_test(tb_path, top_name, output_name):
    vvp_path = OUTPUT_DIR / output_name

    compile_result = subprocess.run(
        [
            "iverilog",
            "-g2012",
            "-s", top_name,
            "-o", str(vvp_path),
            str(buggy_rtl_path),
            str(tb_path),
        ],
        capture_output=True,
        text=True
    )

    print(f"\n=== {top_name} COMPILE ===")
    print("return code:", compile_result.returncode)
    print(compile_result.stdout)
    print(compile_result.stderr)

    if compile_result.returncode == 0:
        sim_result = subprocess.run(
            ["vvp", str(vvp_path)],
            capture_output=True,
            text=True
        )

        print(f"=== {top_name} SIMULATION ===")
        print(sim_result.stdout)
        print(sim_result.stderr)

        return sim_result.stdout

    return None


baseline_bug_output = compile_and_run_bug_test(
    baseline_bug_tb,
    "alu_tb_bug_baseline",
    "bug_baseline.vvp"
)

optimized_bug_output = compile_and_run_bug_test(
    optimized_bug_tb,
    "alu_tb_bug_optimized",
    "bug_optimized.vvp"
)


=== alu_tb_bug_baseline COMPILE ===
return code: 0


=== alu_tb_bug_baseline SIMULATION ===
FAIL test 12: a=128 b=7 op=8 | got y=1 z=0 n=0 c=0 v=0 | expected y=255 z=0 n=1 c=0 v=0
TOTAL FAILURES: 1
alu_coverage_output/alu_tb_bug_baseline.v:318: $finish called at 160000 (1ps)



=== alu_tb_bug_optimized COMPILE ===
return code: 0


=== alu_tb_bug_optimized SIMULATION ===
FAIL test 12: a=128 b=7 op=8 | got y=1 z=0 n=0 c=0 v=0 | expected y=255 z=0 n=1 c=0 v=0
FAIL test 27: a=128 b=1 op=8 | got y=64 z=0 n=0 c=0 v=0 | expected y=192 z=0 n=1 c=0 v=0
TOTAL FAILURES: 2
alu_coverage_output/alu_tb_bug_optimized.v:912: $finish called at 490000 (1ps)




### Generate Final Correct-DUT Testbench
Create the final self-checking testbench using all 49 optimized patterns and the correct ALU.

In [92]:
# Final self-checking testbench using the CORRECT DUT

final_tb_path = OUTPUT_DIR / "alu_testbench_final.v"

make_self_checking_tb(
    optimized_with_golden,
    "alu_testbench_final",
    final_tb_path,
    Path("coverage_guided_alu.v")
)

print("Generated:", final_tb_path)

Generated: alu_coverage_output/alu_testbench_final.v


### Run Final Verification
Compile and simulate the correct DUT and confirm that all optimized patterns pass with zero failures.

In [93]:
final_vvp_path = OUTPUT_DIR / "alu_final.vvp"

compile_result = subprocess.run(
    [
        "iverilog",
        "-g2012",
        "-s", "alu_testbench_final",
        "-o", str(final_vvp_path),
        "coverage_guided_alu.v",
        str(final_tb_path),
    ],
    capture_output=True,
    text=True
)

print("=== FINAL COMPILE ===")
print("return code:", compile_result.returncode)
print(compile_result.stdout)
print(compile_result.stderr)

if compile_result.returncode == 0:
    sim_result = subprocess.run(
        ["vvp", str(final_vvp_path)],
        capture_output=True,
        text=True
    )

    print("=== FINAL SIMULATION ===")
    print(sim_result.stdout)
    print(sim_result.stderr)

=== FINAL COMPILE ===
return code: 0


=== FINAL SIMULATION ===
TOTAL FAILURES: 0
alu_coverage_output/alu_testbench_final.v:912: $finish called at 490000 (1ps)


